In [1]:
#| default_exp models.pesh

In [2]:
#| export

from __future__ import annotations
from typing import List, Dict, Optional, Callable, Union
import numpy as np
import pandas as pd
import copy
from peshbeen.model_selection import SplitTimeSeries
# dot not show warnings
import warnings
warnings.filterwarnings("ignore")
import copy
from scipy.optimize import minimize

class pesh:
    
    def __init__(
        self,
        models: dict,
        weighting_scheme: Optional[Dict[str, float]] = None
        ) -> None:

        """
        Initialize the pesh model with the specified parameters for hybrid forecasting that combines forecasts from multiple models.

        Parameters
        ----------
        models : dict
            A dictionary of model instances to be used for forecasting. The keys should be string names for each model.

        weighting_scheme : dict | None, default None
            Optional dictionary specifying weights for each model's forecast. Default is None, which means equal weighting.

        Returns
        -------
        None
        """

        self.models = models
        self.weighting_scheme = weighting_scheme
    # ─────────────────────────────────────────────────────────────────────────
    # FIT
    # ─────────────────────────────────────────────────────────────────────────

    def fit(self,
            df: pd.DataFrame
            ) -> None:
        
        """
        Fit the specified models to the training data.

        Parameters
        ----------
        df : pd.DataFrame
            Training DataFrame containing the target and any feature columns.
        """

        for model in self.models.values():
            model.fit(df)
            
    def copy(self):
        return copy.deepcopy(self)

    # ─────────────────────────────────────────────────────────────────────────
    # FORECAST
    # ─────────────────────────────────────────────────────────────────────────

    def forecast(
        self,
        H: int,
        exog: Optional[pd.DataFrame] = None,
    ) -> np.ndarray:
        """
        Recursive multi-step forecast.

        Parameters
        ----------
        H : int
            Forecast horizon.
        exog : pd.DataFrame | None, default None
            Optional dataframe of future regressors. Must have the same columns as the exogenous variables used during training and at least `H` rows.

        Returns
        -------
        np.ndarray
            Forecast values of length `H`.
        """
        forecasts = {name: model.forecast(H=H, exog=exog) if exog is not None else model.forecast(H=H) for name, model in self.models.items()}
        # add mean of forecasts as final forecast add as as hybrid forecast to dictionary
        if self.weighting_scheme is not None:
            # check if all models have weights specified, if not raise error
            if not all(name in self.weighting_scheme for name in self.models.keys()):
                raise ValueError("All models must have weights specified in the weighting_scheme dictionary.")
            # check if weights sum to 1, if not raise error
            if not np.isclose(sum(self.weighting_scheme.values()), 1):
                raise ValueError("Weights in the weighting_scheme dictionary must sum to 1.")
            forecasts["pesh"] = np.sum([self.weighting_scheme[name] * forecasts[name] for name in self.models.keys()], axis=0)
        else:
            forecasts["pesh"] = np.mean(list(forecasts.values()), axis=0)
        
        return pd.DataFrame(forecasts)

    def cross_validate(
        self,
        df: pd.DataFrame,
        cv_split: int,
        test_size: int,
        metrics: List[Callable],
        step_size: int = 1,
        metric_to_opt: Optional[Callable] = None,
        weighting_scheme: Optional[Union[Dict[str, float], str]] = None,
        optimizer: str = "SLSQP"
    ) -> pd.DataFrame:
        
        """
        Perform cross-validation for the pesh model using a rolling forecasting origin approach.

        Parameters
        ----------
        df : pd.DataFrame
            The input DataFrame containing the target and any feature columns.
        cv_split : int
            The number of cross-validation splits.
        test_size : int
            The size of the test set for each split.
        metrics : list of callable
            Metric functions (e.g. ``[MAE, RMSE]``) used to evaluate forecast accuracy across folds. Call ``.cv_summary()`` after cross-validation to retrieve the aggregated scores.
        step_size : int, default 1
            The step size for rolling the forecasting origin.
        metric_to_opt : callable, optional
            An optional metric function to optimize when weighting_scheme is set to "optimize". If None, it defaults to the first metric in the metrics list.
        weighting_scheme : dict | str | None, default None                                  
            None: equal weights across models. dict: user-provided weights (must sum to 1). "optimize": optimize weights to minimize MSE via `scipy.optimize.minimize`.
        optimizer : str, default "SLSQP"
            Optimization method to use when weighting_scheme is set to "optimize". Passed to `scipy.optimize.minimize`. Refer to SciPy documentation for available methods.

        Returns
        -------
        pd.DataFrame
            A DataFrame containing the performance metrics for each model and the combined forecast across all cross-validation splits. Also, optimized weights are stored in `self.optimal_weights_` if `weighting_scheme` is "optimize".
        """

        if not isinstance(metrics, list):
            metrics = [metrics]
        else:
            if len(metrics) == 0:
                raise ValueError("At least one metric must be provided in the metrics list.")
        cv_df_ = pd.DataFrame()
        self.forecast_res = {name: np.array([]) for name in self.models.keys()}

        tscv = SplitTimeSeries(n_splits=cv_split, test_size=test_size, step_size=step_size)
        target_col = list(self.models.values())[0].target_col

        for idx, (train_index, test_index) in enumerate(tscv.split(df)):
            train, test = df.iloc[train_index], df.iloc[test_index]
            x_test = test.drop(columns=[target_col], errors="ignore")
            y_test = test[target_col].to_numpy()

            for model in self.models.values():
                model.fit(train)

            exog_test = x_test if x_test.shape[1] > 0 else None
            fold_forecasts = self.forecast(H=test_size, exog=exog_test)

            for name in self.models.keys():
                self.forecast_res[name] = np.concatenate([self.forecast_res[name], fold_forecasts[name]])

            split_results = {
                "cutoff": np.repeat(test.index[0], len(test)),
                "index": test.index,
                "split": np.repeat(f"fold_{idx + 1}", len(test)),
                "y_true": y_test,
            }
            split_results.update(fold_forecasts)
            cv_df_ = pd.concat([cv_df_, pd.DataFrame(split_results)], ignore_index=True)

        model_cols = list(self.models.keys())
        y_values = cv_df_["y_true"].to_numpy()
        for_vals = cv_df_[model_cols].to_numpy()
        n_models = for_vals.shape[1]

        if weighting_scheme is None:
            weights = np.full(n_models, 1.0 / n_models)
            self.optimal_weights_ = dict(zip(model_cols, weights))

        elif isinstance(weighting_scheme, dict):
            if not all(name in weighting_scheme for name in model_cols):
                raise ValueError("All models must have weights specified in weighting_scheme.")
            if not np.isclose(sum(weighting_scheme.values()), 1.0):
                raise ValueError("Weights in weighting_scheme must sum to 1.")
            weights = np.array([weighting_scheme[name] for name in model_cols], dtype=float)
            self.optimal_weights_ = dict(zip(model_cols, weights))

        elif weighting_scheme == "optimize":

            if metric_to_opt is None:
                metric_to_opt = metrics[0]
            def objective(w, forecasts, y_true):
                return metric_to_opt(y_true, forecasts @ w)

            initial_weights = np.full(n_models, 1.0 / n_models)
            constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
            bounds = [(0.0, 1.0)] * n_models

            result = minimize(
                objective,
                initial_weights,
                args=(for_vals, y_values),
                method=optimizer,
                bounds=bounds,
                constraints=constraints,
            )
            weights = result.x if result.success else initial_weights
            self.optimal_weights_ = dict(zip(model_cols, weights))

        else:
            raise ValueError("weighting_scheme must be None, a dict, or 'optimize'.")

        cv_df_["pesh"] = for_vals @ weights
        self.forecast_res["pesh"] = cv_df_["pesh"].to_numpy()

        model_cols.append("pesh")

        perf = {col: [] for col in model_cols}
        for col in model_cols:
            for metric in metrics:
                perf[col].append(metric(y_values, cv_df_[col].to_numpy()))
                
        perf= pd.DataFrame(perf)
        perf.index = [m.__name__ for m in metrics]

        self.cv_summary = perf
        return cv_df_
    
    # a name for the class that is more descriptive of its purpose
    def get_name(self):
        return "pesh"

In [3]:
#| hide

from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown="ignore")
from peshbeen.models import arima, ml_forecaster
from peshbeen.datasets import load_wales_admissions
from peshbeen.metrics import MAE, RMSE
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression
# import RandomForestRegressor from sklearn
from sklearn.ensemble import RandomForestRegressor
wales_admissions = load_wales_admissions()

wales_admissions["day_of_week"] = wales_admissions.index.dayofweek
wales_admissions["month"] = wales_admissions.index.month
# split the data into train and test sets
train = wales_admissions[:-30]
test = wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
from sklearn.linear_model import LinearRegression
lgb_ = ml_forecaster(model=LGBMRegressor(n_estimators=100, verbose=-1),
              target_col='admissions', lags = 7, categorical_encoder=ohe,
              cat_variables=cat_variables, trend = 'ets', ets_params={'trend': 'add', 'seasonal': 'add', 'seasonal_periods': 7})
xgb_ = ml_forecaster(model=XGBRegressor(n_estimators=100),
              target_col='admissions', lags = 7, categorical_encoder=ohe,
              cat_variables=cat_variables, trend = 'ets', ets_params={'trend': 'add', 'seasonal': 'add', 'seasonal_periods': 7})

rf_ = ml_forecaster(model=RandomForestRegressor(n_estimators=100, random_state=42),
              target_col='admissions', lags = 7, categorical_encoder=ohe,
              cat_variables=cat_variables, trend = 'ets', ets_params={'trend': 'add', 'seasonal': 'add', 'seasonal_periods': 7})
lr_ = ml_forecaster(model=LinearRegression(),
              target_col='admissions', lags = 7, categorical_encoder=ohe,
              cat_variables=cat_variables, trend = 'ets', ets_params={'trend': 'add', 'seasonal': 'add', 'seasonal_periods': 7})

arima_model = arima(order=(1,1,1), seasonal_order=(1,1,1), target_col='admissions',
                    categorical_encoder=ohe, cat_variables=cat_variables)
models = {"lgb": lgb_, "xgb": xgb_, "rf": rf_, "lr": lr_, "arima": arima_model}
# models = {"lgb": lgb_, "xgb": xgb_, "rf": rf_, "lr": lr_}
pesh_model = pesh(models=models)
perf_df = pesh_model.cross_validate(df=train, cv_split=5, test_size=30, step_size=30, metrics=[MAE, RMSE], metric_to_opt=RMSE, weighting_scheme="optimize")


In [4]:
#| hide

opt_weights = pesh_model.optimal_weights_
pesh_model = pesh(models=models, weighting_scheme=opt_weights)
pesh_model.fit(train)
pesh_forecasts = pesh_model.forecast(H=30, exog=test[cat_variables])
